# Results — DA-Price Ablation Campaign (DK1)

Read-only visualisation of the ablation campaign artefacts.

**Campaign matrix** — 8 CPU specs × 4 feature-set variants × 3 modes = 96 runs, plus 5 tuneless baselines and (when available) 4 GPU TabNet runs. Core artefacts are provided in `../artifacts/` and were produced by `da-experiments-fs4-rerun.ipynb`; this notebook never writes to the campaign cache.

Re-run the notebook to refresh figures as TabNet variants land.

## 1 — Setup and data load

In [ ]:
import json
from pathlib import Path

# Package-relative paths for the cleaned public copy.
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PACKAGE_ROOT = NOTEBOOK_DIR.parent
else:
    PACKAGE_ROOT = NOTEBOOK_DIR
ARTIFACTS_DIR = PACKAGE_ROOT / "artifacts"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
})

CACHE = ARTIFACTS_DIR
FIG_DIR = PACKAGE_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_parquet(CACHE / 'results_df.parquet')
preds   = pd.read_parquet(CACHE / 'test_predictions_df.parquet')

print(f'results_df : {results.shape[0]} rows × {results.shape[1]} cols')
print(f'preds      : {preds.shape[0]} rows × {preds.shape[1]} cols (test window {preds.index.min()} → {preds.index.max()})')

BASELINES = {'price_baseline_naive_d1', 'price_baseline_naive_w1', 'price_baseline_naive_epf_hybrid', 'climatology_how', 'lear'}
results['is_baseline'] = results['run_name'].isin(BASELINES)

miss_model = results['model'].isna() & ~results['is_baseline']
if miss_model.any():
    results.loc[miss_model, 'model'] = results.loc[miss_model, 'run_name'].str.split('_').str[0]

tuned = results[~results['is_baseline']].copy()
baselines = results[results['is_baseline']].copy()

FS_ORDER     = ['fs1', 'fs2', 'fs3', 'fs4']
MODE_ORDER   = ['fixed', 'no_penalty', 'with_penalty']
SPEC_ORDER   = ['xgb', 'lgbm', 'catboost', 'hgbt', 'rf', 'ridge', 'lasso', 'enet', 'tabnet']
FAMILY_ORDER = ['tree', 'linear', 'neural']

FEATURE_SET_VARIANTS = {
    'fs1': {'time'},
    'fs2': {'time', 'price'},
    'fs3': {'time', 'price', 'fundamentals'},
    'fs4': {'time', 'price', 'fundamentals', 'weather'},
}
FAMILY_COLOR = {
    'time':         '#7BA9D9',
    'price':        '#E08D6C',
    'fundamentals': '#9DC58A',
    'weather':      '#C58AD9',
}

print(f'\ntuned: {len(tuned)} rows  ·  baselines: {len(baselines)} rows')
print('specs:', sorted(set(tuned['model'].dropna())))
print('modes:', sorted(set(tuned['ablation_mode'].dropna())))

## 2 — Headline table

All runs sorted by test MAE (EUR/MWh) on the held-out 28-day window 2026-03-18 → 2026-04-14.

In [ ]:
headline_cols = ['run_name', 'family', 'feature_variant', 'ablation_mode',
                 'test_mae', 'test_rmse', 'test_r2',
                 'mean_fold_val_mae', 'n_selected_groups', 'n_selected_features',
                 'study_wall_seconds']
headline = results[headline_cols].copy().sort_values('test_mae').reset_index(drop=True)
headline.index = headline.index + 1
headline.index.name = 'rank'
headline

## 3 — Three-mode ablation comparison

Per `(spec, FS)`: test-MAE delta between **fixed** vs **no_penalty** vs **with_penalty**.

In [ ]:
wide_mae = (tuned
    .pivot_table(index=['model', 'feature_variant'],
                 columns='ablation_mode', values='test_mae')
    .reindex(columns=MODE_ORDER)
)
wide_mae['Δ(np−fixed)'] = wide_mae['no_penalty']   - wide_mae['fixed']
wide_mae['Δ(wp−fixed)'] = wide_mae['with_penalty'] - wide_mae['fixed']
wide_mae['Δ(wp−np)']    = wide_mae['with_penalty'] - wide_mae['no_penalty']
wide_mae['best_mode']   = wide_mae[MODE_ORDER].idxmin(axis=1)
wide_mae = wide_mae.loc[
    [(s,f) for s in SPEC_ORDER for f in FS_ORDER if (s,f) in wide_mae.index]
]
wide_mae

In [ ]:
win_counts = wide_mae['best_mode'].value_counts().reindex(MODE_ORDER).fillna(0).astype(int)
print('Best mode by (spec, FS) pair (n=' + str(int(win_counts.sum())) + '):')
for m, c in win_counts.items():
    print(f'  {m:14s}: {c}')
print()
print('Mean Δ test-MAE (EUR/MWh) vs fixed mode:')
print(f"  no_penalty   − fixed: {wide_mae['Δ(np−fixed)'].mean():+.3f}  (median {wide_mae['Δ(np−fixed)'].median():+.3f})")
print(f"  with_penalty − fixed: {wide_mae['Δ(wp−fixed)'].mean():+.3f}  (median {wide_mae['Δ(wp−fixed)'].median():+.3f})")
print(f"  with_penalty − np   : {wide_mae['Δ(wp−np)'].mean():+.3f}  (median {wide_mae['Δ(wp−np)'].median():+.3f})")

## 4 — Heatmap of test MAE across `(spec × FS × mode)`

Cell colour = test MAE in EUR/MWh; lower is better. Three columns per spec lets you read the three-mode story at a glance.

In [ ]:
specs_present = [s for s in SPEC_ORDER if s in tuned['model'].unique()]
rows_idx, values = [], []
for spec in specs_present:
    for fs in FS_ORDER:
        rows_idx.append(f'{spec}_{fs}')
        row = []
        for mode in MODE_ORDER:
            sel = tuned[(tuned['model']==spec) & (tuned['feature_variant']==fs) & (tuned['ablation_mode']==mode)]
            row.append(sel['test_mae'].iloc[0] if len(sel) else np.nan)
        values.append(row)
heat = pd.DataFrame(values, index=rows_idx, columns=MODE_ORDER)

fig, ax = plt.subplots(figsize=(7, max(6, 0.32*len(heat))))
vmin = np.nanmin(heat.values); vmax = np.nanmax(heat.values)
im = ax.imshow(heat.values, aspect='auto', cmap='viridis_r', vmin=vmin, vmax=vmax)
ax.set_xticks(range(len(MODE_ORDER))); ax.set_xticklabels(MODE_ORDER)
ax.set_yticks(range(len(rows_idx))); ax.set_yticklabels(rows_idx, fontsize=8)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if np.isfinite(v):
            txt_color = 'white' if (v - vmin) / max(vmax - vmin, 1e-9) > 0.55 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7.5, color=txt_color)
ax.set_title('Test MAE (EUR/MWh) — spec × feature-set × mode')
fig.colorbar(im, ax=ax, label='test MAE (EUR/MWh)')
ax.grid(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_heatmap_test_mae.png', bbox_inches='tight')
plt.show()

## 5 — Feature-set effect (FS1 → FS4) per spec, mode = no_penalty

Isolates the **quantity of feature engineering**: does adding price (FS2), fundamentals (FS3), or weather (FS4) lower test MAE?

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
cmap_specs = plt.get_cmap('tab10')
for i, spec in enumerate(specs_present):
    sub = tuned[(tuned['model']==spec) & (tuned['ablation_mode']=='no_penalty')].set_index('feature_variant')
    if sub.empty: continue
    y = [sub.loc[fs, 'test_mae'] if fs in sub.index else np.nan for fs in FS_ORDER]
    ax.plot(FS_ORDER, y, marker='o', linewidth=1.5, label=spec, color=cmap_specs(i % 10))
ax.set_xlabel('feature-set variant')
ax.set_ylabel('test MAE (EUR/MWh)')
ax.set_title('Feature-set effect — mode = no_penalty')
ax.legend(ncol=3, fontsize=9, loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_fs_effect_no_penalty.png', bbox_inches='tight')
plt.show()

np_mode = tuned[tuned['ablation_mode']=='no_penalty'].pivot(index='model', columns='feature_variant', values='test_mae')
np_mode = np_mode.reindex(columns=FS_ORDER)
deltas = pd.DataFrame({
    'Δ(fs2−fs1)': np_mode['fs2'] - np_mode['fs1'],
    'Δ(fs3−fs2)': np_mode['fs3'] - np_mode['fs2'],
    'Δ(fs4−fs3)': np_mode['fs4'] - np_mode['fs3'],
    'Δ(fs4−fs1)': np_mode['fs4'] - np_mode['fs1'],
})
print('\nPer-spec ΔMAE when adding each feature family (no_penalty, EUR/MWh):')
print(deltas.round(3))
print('\nMean across specs:')
print(deltas.mean().round(3))

## 6 — Mode effect per spec (averaged over FS variants)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
agg = (tuned
       .groupby(['model','ablation_mode'])['test_mae'].mean()
       .unstack('ablation_mode')
       .reindex(columns=MODE_ORDER)
)
for i, spec in enumerate(specs_present):
    if spec not in agg.index: continue
    y = agg.loc[spec].values
    ax.plot(MODE_ORDER, y, marker='o', linewidth=1.5, label=spec, color=cmap_specs(i % 10))
ax.set_xlabel('ablation mode')
ax.set_ylabel('mean test MAE (EUR/MWh)  —  averaged over FS1..FS4')
ax.set_title('Ablation mode effect per spec')
ax.legend(ncol=3, fontsize=9, loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_mode_effect_per_spec.png', bbox_inches='tight')
plt.show()

## 7 — Family rollups

Test MAE distribution by model family, faceted by FS variant.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, len(FS_ORDER), figsize=(12, 4), sharey=True)
for ax, fs in zip(axes, FS_ORDER):
    sub = tuned[tuned['feature_variant']==fs]
    if sub.empty:
        ax.set_title(f'{fs}\n(no data)'); continue
    fams = [f for f in FAMILY_ORDER if f in sub['family'].unique()]
    data = [sub[sub['family']==f]['test_mae'].values for f in fams]
    bp = ax.boxplot(data, labels=fams, patch_artist=True, widths=0.55)
    fam_colors = {'tree':'#9DCDF0','linear':'#F0B79D','neural':'#C9F09D'}
    for patch, fam in zip(bp['boxes'], fams):
        patch.set_facecolor(fam_colors.get(fam, '#cccccc'))
        patch.set_alpha(0.85)
    for fam, x in zip(fams, range(1, len(fams)+1)):
        vals = sub[sub['family']==fam]['test_mae'].values
        if len(vals) == 0: continue
        jitter = rng.uniform(-0.06, 0.06, size=len(vals))
        ax.scatter(np.full_like(vals, x, dtype=float) + jitter, vals,
                   color='black', s=10, alpha=0.55, zorder=3)
    ax.set_title(fs)
    ax.set_ylabel('test MAE (EUR/MWh)' if fs == FS_ORDER[0] else '')
fig.suptitle('Test MAE distribution by family, per FS', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_family_distribution.png', bbox_inches='tight')
plt.show()

## 8 — Penalty effect on parsimony

In [ ]:
alg_modes = ['no_penalty', 'with_penalty']
parsim = (tuned[tuned['ablation_mode'].isin(alg_modes)]
          .groupby(['model','feature_variant','ablation_mode'])[['n_selected_groups','n_selected_features']]
          .first()
          .unstack('ablation_mode'))
parsim.columns = [f'{a}__{b}' for a, b in parsim.columns]
if 'n_selected_groups__no_penalty' in parsim and 'n_selected_groups__with_penalty' in parsim:
    parsim['Δ_groups (wp−np)']   = parsim['n_selected_groups__with_penalty']   - parsim['n_selected_groups__no_penalty']
if 'n_selected_features__no_penalty' in parsim and 'n_selected_features__with_penalty' in parsim:
    parsim['Δ_features (wp−np)'] = parsim['n_selected_features__with_penalty'] - parsim['n_selected_features__no_penalty']
parsim_show = parsim.reindex([(s,f) for s in SPEC_ORDER for f in FS_ORDER if (s,f) in parsim.index])
print('Selected counts (no_penalty vs with_penalty) and deltas:')
parsim_show

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for mode_lbl, color in [('no_penalty', '#1f77b4'), ('with_penalty', '#d62728')]:
    col = f'n_selected_groups__{mode_lbl}'
    if col not in parsim.columns: continue
    series = parsim[col].dropna()
    if series.empty: continue
    ax[0].hist(series, bins=range(int(series.min()), int(series.max())+2), alpha=0.6, label=mode_lbl, color=color)
ax[0].set_xlabel('# groups selected'); ax[0].set_ylabel('count')
ax[0].set_title('Selected groups: no_penalty vs with_penalty'); ax[0].legend()
for mode_lbl, color in [('no_penalty', '#1f77b4'), ('with_penalty', '#d62728')]:
    col = f'n_selected_features__{mode_lbl}'
    if col not in parsim.columns: continue
    series = parsim[col].dropna()
    if series.empty: continue
    ax[1].hist(series, bins=20, alpha=0.6, label=mode_lbl, color=color)
ax[1].set_xlabel('# features selected'); ax[1].set_ylabel('count')
ax[1].set_title('Selected features: no_penalty vs with_penalty'); ax[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_penalty_parsimony.png', bbox_inches='tight')
plt.show()

## 9 — Group-selection frequency, candidate-corrected

Raw inclusion counts overstate time / price groups because those families are candidates in more `(spec, FS, mode)` cells than fundamentals or weather. The *relative* frequency divides selections by the number of cells in which the group's family was actually a candidate, so weather and fundamentals groups are evaluated on equal footing with time and price.

Candidate exposure per family (across the 64 algorithmic cells):
- **time** — 64 cells (fs1..fs4, both algo modes)
- **price** — 48 cells (fs2..fs4)
- **fundamentals** — 32 cells (fs3..fs4)
- **weather** — 16 cells (fs4 only)

In [ ]:
def parse_groups(s):
    if pd.isna(s) or s in ('', '[]'): return []
    if isinstance(s, (list, tuple)): return list(s)
    if isinstance(s, str) and ',' in s:
        return [t.strip() for t in s.split(',') if t.strip()]
    try:
        return json.loads(s)
    except Exception:
        return []

def split_family(g):
    if '__' in g:
        fam, grp = g.split('__', 1)
        return fam, grp
    return None, g

tuned_alg = tuned[tuned['ablation_mode'].isin(alg_modes)].copy()
tuned_alg['groups_list'] = tuned_alg['selected_groups_str'].apply(parse_groups)
n_alg_cells = len(tuned_alg)

all_groups = sorted({g for lst in tuned_alg['groups_list'] for g in lst})
all_groups = [g for g in all_groups if g != 'lear-features']

# selected counts
sel_count = pd.Series(
    [g for lst in tuned_alg['groups_list'] for g in lst if g in all_groups]
).value_counts()

# candidate counts: a group is a candidate in any algorithmic cell whose FS contains the group's family
cand_count = {}
for g in all_groups:
    fam, _ = split_family(g)
    if fam is None: continue
    cand_count[g] = int(((tuned_alg['feature_variant'].apply(lambda fs: fam in FEATURE_SET_VARIANTS[fs]))).sum())

freq_corrected = pd.DataFrame({
    'family':       [split_family(g)[0] for g in all_groups],
    'group':        [split_family(g)[1] for g in all_groups],
    'selected':     [int(sel_count.get(g, 0)) for g in all_groups],
    'candidate':    [cand_count.get(g, 0) for g in all_groups],
}, index=all_groups)
freq_corrected['relative_freq']  = freq_corrected['selected'] / freq_corrected['candidate'].replace(0, np.nan)
freq_corrected['raw_pct_of_alg'] = freq_corrected['selected'] / max(n_alg_cells, 1) * 100
freq_corrected = freq_corrected.sort_values('relative_freq', ascending=False)

print(f'Total algorithmic cells: {n_alg_cells}')
print(f'Distinct candidate groups across all cells: {len(freq_corrected)}')
print('\nFamily totals (selected / candidate-cells):')
fam_summary = (freq_corrected.groupby('family')[['selected','candidate']].sum())
fam_summary['mean_relative_freq'] = (freq_corrected.groupby('family')['relative_freq'].mean())
print(fam_summary.round(3))
print('\nTop 25 by relative frequency:')
freq_corrected.head(25)

In [ ]:
fc = freq_corrected.copy().sort_values('relative_freq', ascending=True)
fig, ax = plt.subplots(figsize=(11, max(8, 0.22 * len(fc))))
y = np.arange(len(fc))
colors = [FAMILY_COLOR.get(fam, '#888888') for fam in fc['family']]
ax.barh(y, fc['relative_freq']*100, color=colors, alpha=0.9)
ax.set_yticks(y)
ax.set_yticklabels([f"{fam} · {grp}" for fam, grp in zip(fc['family'], fc['group'])], fontsize=7.5)
ax.set_xlabel('relative frequency = selected / candidate-cells (%)')
ax.set_xlim(0, 100)
ax.set_title('Group-selection frequency, corrected for candidate exposure')
for i, (sel, cand, rf) in enumerate(zip(fc['selected'], fc['candidate'], fc['relative_freq'])):
    ax.text(rf*100 + 0.5, i, f"{int(sel)}/{int(cand)}", va='center', fontsize=7.5)
legend_handles = [Patch(facecolor=FAMILY_COLOR[f], label=f) for f in ['time','price','fundamentals','weather'] if f in FAMILY_COLOR]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_group_frequency_corrected.png', bbox_inches='tight')
plt.show()

## 10 — Validation- vs test-MAE alignment

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
fam_colors = {'tree':'#1f77b4','linear':'#d62728','neural':'#2ca02c'}
mode_marker = {'fixed':'o','no_penalty':'^','with_penalty':'s'}
for fam in FAMILY_ORDER:
    for mode in MODE_ORDER:
        sub = tuned[(tuned['family']==fam) & (tuned['ablation_mode']==mode)]
        if sub.empty: continue
        ax.scatter(sub['mean_fold_val_mae'], sub['test_mae'],
                   color=fam_colors.get(fam,'gray'), marker=mode_marker[mode],
                   alpha=0.75, s=45, label=f'{fam} / {mode}')
lims_lo = min(tuned['mean_fold_val_mae'].min(), tuned['test_mae'].min())
lims_hi = max(tuned['mean_fold_val_mae'].max(), tuned['test_mae'].max())
pad = 0.05 * (lims_hi - lims_lo)
ax.plot([lims_lo - pad, lims_hi + pad], [lims_lo - pad, lims_hi + pad], color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('mean fold-val MAE (EUR/MWh)')
ax.set_ylabel('test MAE (EUR/MWh)')
ax.set_title('CV-val vs test alignment')
ax.legend(fontsize=8, ncol=2, loc='best')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_cv_vs_test.png', bbox_inches='tight')
plt.show()
print(f"\nCorrelation (val vs test, MAE): {tuned[['mean_fold_val_mae','test_mae']].corr().iloc[0,1]:.3f}")
print(f"Mean signed gap (test − val):    {(tuned['test_mae'] - tuned['mean_fold_val_mae']).mean():+.3f}")

## 11 — Search wall-time diagnostics

In [ ]:
wall = tuned[['family','model','feature_variant','ablation_mode','study_wall_seconds','per_trial_seconds','final_fit_seconds']].copy()
wall['study_wall_min'] = wall['study_wall_seconds'] / 60.0
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fams = [f for f in FAMILY_ORDER if f in wall['family'].unique()]
data1 = [wall[wall['family']==f]['study_wall_min'].dropna().values for f in fams]
axes[0].boxplot(data1, labels=fams, patch_artist=True, widths=0.55)
axes[0].set_ylabel('study wall time (min)'); axes[0].set_title('Per-cell Optuna study wall time')
data2 = [wall[wall['family']==f]['per_trial_seconds'].dropna().values for f in fams]
axes[1].boxplot(data2, labels=fams, patch_artist=True, widths=0.55)
axes[1].set_ylabel('per-trial wall time (s)'); axes[1].set_title('Per-trial wall time')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_search_walltime.png', bbox_inches='tight')
plt.show()
print('\nWall-time summary by family:')
wall.groupby('family')[['study_wall_min','per_trial_seconds','final_fit_seconds']].agg(['mean','median','max']).round(1)

## 12 — Test-window prediction trace (best vs LEAR vs actual)

In [ ]:
best_run = tuned.sort_values('test_mae').iloc[0]['run_name']
print(f'Best tuned run: {best_run} (test MAE = {tuned[tuned.run_name==best_run].test_mae.iloc[0]:.3f} EUR/MWh)')
if 'lear' in results['run_name'].values:
    print(f"LEAR baseline:          test MAE = {results[results.run_name=='lear'].test_mae.iloc[0]:.3f} EUR/MWh")

# Split the test window into two halves so individual price events
# become visible. The cutoff is the midpoint of the test index.
mid = preds.index[len(preds.index) // 2]
top_idx = preds.index[preds.index < mid]
bot_idx = preds.index[preds.index >= mid]

fig, axes = plt.subplots(2, 1, figsize=(13, 7.5), sharey=True)
for ax, idx in zip(axes, [top_idx, bot_idx]):
    ax.plot(idx, preds.loc[idx, 'actual'], label='actual',
            color='black', linewidth=1.0, alpha=0.95)
    if best_run in preds.columns:
        ax.plot(idx, preds.loc[idx, best_run], label=f'best: {best_run}',
                color='#1f77b4', linewidth=1.0, alpha=0.9)
    if 'lear' in preds.columns:
        ax.plot(idx, preds.loc[idx, 'lear'], label='LEAR',
                color='#d62728', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('DK1 DA price (EUR/MWh)')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(idx.min(), idx.max())
axes[0].set_title(f'Test-window predictions \u2014 {preds.index.min().date()} to {preds.index.max().date()}')
axes[0].legend(loc='upper right', framealpha=0.9)
axes[1].set_xlabel('valid_from (UTC)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_test_trace_best_vs_baseline.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ----- Test window: realised price + every baseline (3x2 small multiples) -----
# Used as the opening figure of the Results chapter: shows what we are
# trying to predict, and how each standard baseline performs on the same
# 28-day window. 3 rows x 2 cols gives each panel enough vertical room
# to show daily structure clearly.

import matplotlib.dates as mdates

baseline_specs = [
    ('lear',                            'LEAR',                '#2ca02c'),
    ('price_baseline_naive_d1',         'Persistence 24h',     '#d62728'),
    ('price_baseline_naive_w1',         'Persistence 168h',    '#9467bd'),
    ('price_baseline_naive_epf_hybrid', 'Na\u00efve EPF hybrid', '#ff7f0e'),
    ('climatology_how',                 'Climatology (HoW)',   '#8c564b'),
]
present = [(c, lab, col) for c, lab, col in baseline_specs if c in preds.columns]
print(f'Baselines present: {[lab for _, lab, _ in present]}')

# 3 rows x 2 cols: actual alone in top-left, then 5 baseline panels
fig, axes = plt.subplots(3, 2, figsize=(13, 9.5), sharex=True, sharey=True)
axes_flat = list(axes.flatten())

# Panel 1: actual alone
ax0 = axes_flat[0]
ax0.plot(preds.index, preds['actual'], color='black', linewidth=1.0)
ax0.set_title('Realised DK1 day-ahead price', fontsize=11)
ax0.grid(True, alpha=0.3)

# Panels 2..6: actual (grey) + one baseline (colour)
for ax, (col, label, colour) in zip(axes_flat[1:], present):
    ax.plot(preds.index, preds['actual'], color='grey', linewidth=0.7, alpha=0.55, label='actual')
    ax.plot(preds.index, preds[col],     color=colour, linewidth=0.95, alpha=0.95, label=label)
    mae = (preds[col] - preds['actual']).abs().mean()
    ax.set_title(f'{label}  \u2014  MAE = {mae:.2f} EUR/MWh', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.85)

# Hide trailing empty panels (in case fewer than 5 baselines were present)
for ax in axes_flat[1 + len(present):]:
    ax.axis('off')

# Axis labels: only on outer edges
for ax in axes[-1]:
    ax.set_xlabel('valid_from (UTC)')
for ax in axes[:, 0]:
    ax.set_ylabel('Price (EUR/MWh)')

# Tidy date ticks (only the bottom row needs them, but apply to all
# because sharex propagates)
for ax in axes_flat:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
for ax in axes[-1]:
    for label in ax.get_xticklabels():
        label.set_rotation(30); label.set_ha('right')

fig.suptitle(
    f'Test window {preds.index.min().date()} \u2192 {preds.index.max().date()}: '
    f'realised DK1 price and baseline forecasts',
    fontsize=12, y=1.005,
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_test_period_baselines.png', bbox_inches='tight', dpi=150)
plt.show()

## 13 — Best of each family

In [ ]:
best_per_family = (tuned.sort_values('test_mae')
                    .groupby('family', as_index=False).first()
                    [['family','run_name','feature_variant','ablation_mode','test_mae','test_rmse','test_r2','n_selected_groups','n_selected_features','study_wall_seconds']])
best_per_family

## 14 — Baselines vs best tuned

In [ ]:
best_tuned = tuned.sort_values('test_mae').iloc[0]
rows = []
for _, b in baselines.iterrows():
    rows.append({'kind':'baseline', 'name':b['run_name'], 'test_mae':b['test_mae'], 'test_rmse':b['test_rmse']})
rows.append({'kind':'best tuned','name':best_tuned['run_name'],'test_mae':best_tuned['test_mae'],'test_rmse':best_tuned['test_rmse']})
comp = pd.DataFrame(rows).sort_values('test_mae').reset_index(drop=True)
comp

## 15 — rMAE table vs every baseline

For each model run, the **relative MAE** vs each tuneless benchmark — `rMAE_b = test_mae(model) / test_mae(b)`. Values < 1 mean the model beats baseline `b`.

Same denominator set as in `da-experiments.ipynb` §11b: `naive_d1`, `naive_w1`, `epf_hybrid`, `climatology_how`, `lear`.

In [ ]:
BASELINES_FOR_RMAE = ['price_baseline_naive_d1', 'price_baseline_naive_w1',
                       'price_baseline_naive_epf_hybrid', 'climatology_how', 'lear']
SHORT_RMAE = {
    'price_baseline_naive_d1':         'rMAE_naive_d1',
    'price_baseline_naive_w1':         'rMAE_naive_w1',
    'price_baseline_naive_epf_hybrid': 'rMAE_epf_hybrid',
    'climatology_how':                 'rMAE_climatology',
    'lear':                            'rMAE_lear',
}
baseline_mae_lookup = {b: float(results[results['run_name']==b]['test_mae'].iloc[0])
                       for b in BASELINES_FOR_RMAE if b in results['run_name'].values}
print('Baseline test MAEs (EUR/MWh):')
for b, m in baseline_mae_lookup.items():
    print(f'  {b:36s}  {m:8.3f}')
rmae_rows = []
for _, row in results.iterrows():
    out = {
        'run_name':        row['run_name'],
        'family':          row['family'],
        'feature_variant': row['feature_variant'],
        'ablation_mode':   row['ablation_mode'],
        'test_mae':        float(row['test_mae']),
    }
    for b, mae_b in baseline_mae_lookup.items():
        out[SHORT_RMAE[b]] = float(row['test_mae']) / mae_b
    rmae_rows.append(out)
rmae_df = pd.DataFrame(rmae_rows).sort_values('test_mae').reset_index(drop=True)
rmae_df.index = rmae_df.index + 1
rmae_df.index.name = 'rank'
rmae_df

In [ ]:
rmae_cols = [SHORT_RMAE[b] for b in baseline_mae_lookup]
try:
    styled = (rmae_df.style
                .background_gradient(subset=rmae_cols, cmap='RdYlGn_r', vmin=0.3, vmax=1.5)
                .format(precision=3))
    from IPython.display import display
    display(styled)
except Exception as e:
    print(f'(Styler unavailable: {e})')

## 16 — Per-model test traces — actual vs each model

One figure per spec. Each panel = one `(spec, FS, mode)` cell. Two lines per panel: **actual** (black) and **the model's prediction** (blue). Subtitle = test MAE in EUR/MWh.

PNGs saved to `_build/figures/traces/<spec>.png` for thesis use.

In [ ]:
TRACE_DIR = FIG_DIR / 'traces'
TRACE_DIR.mkdir(exist_ok=True)

results_by_run = results.set_index('run_name')

for spec in specs_present:
    fig, axes = plt.subplots(len(FS_ORDER), len(MODE_ORDER),
                              figsize=(13, 1.6 * len(FS_ORDER) + 0.8),
                              sharex=True, sharey=True)
    if len(FS_ORDER) == 1: axes = np.array([axes])
    if len(MODE_ORDER) == 1: axes = axes.reshape(-1, 1)
    for i, fs in enumerate(FS_ORDER):
        for j, mode in enumerate(MODE_ORDER):
            mode_short = {'fixed':'fix','no_penalty':'np','with_penalty':'wp'}[mode]
            run = f'{spec}_{fs}_{mode_short}'
            ax = axes[i, j]
            ax.plot(preds.index, preds['actual'], color='black', linewidth=0.6, alpha=0.85, label='actual')
            if run in preds.columns and run in results_by_run.index:
                ax.plot(preds.index, preds[run], color='#1f77b4', linewidth=0.7, alpha=0.85, label=spec)
                mae = results_by_run.loc[run, 'test_mae']
                ax.set_title(f'{fs} · {mode}  —  MAE={mae:.2f}', fontsize=9)
            else:
                ax.set_title(f'{fs} · {mode}  —  (no run)', fontsize=9, color='#888')
            if i == len(FS_ORDER) - 1:
                ax.tick_params(axis='x', labelrotation=20, labelsize=7)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel('EUR/MWh', fontsize=8)
            ax.tick_params(axis='y', labelsize=7)
            ax.grid(alpha=0.2)
    fig.suptitle(f'Test-window predictions — {spec}', y=1.00, fontsize=12, weight='bold')
    plt.tight_layout()
    plt.savefig(TRACE_DIR / f'{spec}.png', bbox_inches='tight')
    plt.show()

## 17 — Export thesis-ready tables

In [ ]:
TBL_DIR = FIG_DIR / 'tables'
TBL_DIR.mkdir(exist_ok=True)

headline.to_csv(TBL_DIR / 'tbl_headline.csv')
wide_mae.to_csv(TBL_DIR / 'tbl_three_mode.csv')
deltas.to_csv(TBL_DIR / 'tbl_fs_deltas.csv')
parsim_show.to_csv(TBL_DIR / 'tbl_parsimony.csv')
freq_corrected.to_csv(TBL_DIR / 'tbl_group_frequency_corrected.csv')
best_per_family.to_csv(TBL_DIR / 'tbl_best_per_family.csv', index=False)
comp.to_csv(TBL_DIR / 'tbl_baselines_vs_best.csv', index=False)
rmae_df.to_csv(TBL_DIR / 'tbl_rmae.csv')

print('Tables written to', TBL_DIR)
print('Figures written to', FIG_DIR)
print('Per-spec traces written to', TRACE_DIR)
print()
for p in sorted(TBL_DIR.glob('*.csv')) + sorted(FIG_DIR.glob('*.png')) + sorted(TRACE_DIR.glob('*.png')):
    print(f'  {p.relative_to(FIG_DIR.parent)}')

## 18 — Diebold-Mariano significance tests

Pairwise DM tests on the test-window loss differentials, using a Newey-West
HAC long-run variance estimator (Bartlett kernel, lag $L = \max(h-1, \lceil T^{1/3} \rceil)$) with the Harvey-Leybourne-Newbold (1997) finite-sample correction. The reported $p$-value is for the one-sided alternative *"row model has lower loss than column model"* — so a small $p$ in cell $(i, j)$ means model $i$ significantly outperforms model $j$. Three matrices are computed for both absolute and squared loss:

- **Matrix A** — best of each family (LEAR, best linear, best tree, best neural) plus the climatology benchmark.
- **Matrix B** — overall winning spec across feature-set variants $\text{FS}1 \to \text{FS}4$.
- **Matrix C** — overall winning spec across the three ablation modes (`fixed`, `no_penalty`, `with_penalty`).

Pairs that involve NaN predictions (e.g. LEAR's daily-recalibration boundary at the tail of the test window) are computed on the intersection of valid rows; the row count actually used per pair is exported alongside the $p$-value matrices.


In [ ]:
from scipy import stats

TBL_DIR = FIG_DIR / 'tables'
TBL_DIR.mkdir(exist_ok=True)
MODE_SHORT = {'fixed': 'fix', 'no_penalty': 'np', 'with_penalty': 'wp'}


def diebold_mariano(e1, e2, h=1, alternative='two-sided'):
    """DM statistic and p-value for two loss series.

    Newey-West HAC long-run variance with Bartlett kernel and lag
    L = max(h-1, ceil(T**(1/3))). Harvey-Leybourne-Newbold (1997)
    finite-sample correction; reference distribution t(T-1).
    """
    e1 = np.asarray(e1, float).ravel()
    e2 = np.asarray(e2, float).ravel()
    d = e1 - e2
    T = len(d)
    if T < 30:
        return float('nan'), float('nan')
    d_bar = d.mean()
    L = max(h - 1, int(np.ceil(T ** (1 / 3))))
    gamma0 = np.var(d, ddof=0)
    weights = [1 - k / (L + 1) for k in range(1, L + 1)]
    cov_sum = 0.0
    for w, k in zip(weights, range(1, L + 1)):
        gk = np.mean((d[k:] - d_bar) * (d[:-k] - d_bar))
        cov_sum += w * gk
    sigma2 = gamma0 + 2 * cov_sum
    if sigma2 <= 0:
        return float('nan'), float('nan')
    DM = d_bar / np.sqrt(sigma2 / T)
    HLN = DM * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    if alternative == 'two-sided':
        p = 2 * (1 - stats.t.cdf(abs(HLN), df=T - 1))
    elif alternative == 'less':
        p = stats.t.cdf(HLN, df=T - 1)
    elif alternative == 'greater':
        p = 1 - stats.t.cdf(HLN, df=T - 1)
    else:
        raise ValueError(alternative)
    return float(HLN), float(p)


def dm_pairwise_pvalue_matrix(preds_dict, y_true, loss='abs'):
    """Pairwise one-sided DM matrix. Returns (P, N) where
    P.loc[i, j] = P-value for H1: model i has lower loss than j,
    N.loc[i, j] = effective sample size after dropping NaN rows."""
    names = list(preds_dict.keys())
    P = pd.DataFrame(np.full((len(names), len(names)), np.nan), index=names, columns=names)
    N = pd.DataFrame(np.zeros((len(names), len(names)), dtype=int), index=names, columns=names)
    y = pd.Series(y_true).astype(float)
    if loss == 'abs':
        loss_fn = lambda yt, yp: (yt - yp).abs()
    elif loss == 'sq':
        loss_fn = lambda yt, yp: (yt - yp) ** 2
    else:
        raise ValueError(loss)
    series = {k: pd.Series(v).astype(float) for k, v in preds_dict.items()}
    for i in names:
        for j in names:
            if i == j:
                continue
            mask = y.notna() & series[i].notna() & series[j].notna()
            if mask.sum() < 30:
                continue
            ei = loss_fn(y[mask], series[i][mask]).values
            ej = loss_fn(y[mask], series[j][mask]).values
            _, p = diebold_mariano(ei, ej, h=96, alternative='less')
            P.loc[i, j] = p
            N.loc[i, j] = int(mask.sum())
    return P, N


def plot_dm_heatmap(P, ax, title, vmax=0.10):
    n = len(P)
    M = P.values.astype(float)
    im = ax.imshow(M, cmap='RdBu_r', vmin=0, vmax=vmax, aspect='equal')
    for i in range(n):
        for j in range(n):
            v = M[i, j]
            if np.isnan(v):
                ax.text(j, i, '—', ha='center', va='center', fontsize=10, color='#888')
                continue
            txt = f'{v:.3f}' if v >= 0.001 else '<.001'
            color = 'white' if (v < 0.025 or v > 0.95) else 'black'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8, color=color)
    ax.set_xticks(range(n)); ax.set_xticklabels(P.columns, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(n)); ax.set_yticklabels(P.index, fontsize=9)
    ax.set_xlabel('candidate j', fontsize=9)
    ax.set_ylabel('benchmark i  (row better than column)', fontsize=9)
    ax.set_title(title, fontsize=10)
    return im


# Convenience: family-best lookup
best_tree_run    = tuned[tuned['family'] == 'tree'].sort_values('test_mae').iloc[0]['run_name']
best_linear_run  = tuned[tuned['family'] == 'linear'].sort_values('test_mae').iloc[0]['run_name']
neural_present   = (tuned['family'] == 'neural').any()
best_neural_run  = tuned[tuned['family'] == 'neural'].sort_values('test_mae').iloc[0]['run_name'] if neural_present else None
overall_run      = tuned.sort_values('test_mae').iloc[0]['run_name']
print('best tree   :', best_tree_run)
print('best linear :', best_linear_run)
print('best neural :', best_neural_run)
print('overall     :', overall_run)


In [ ]:
# Matrix A — best-of-family + climatology
preds_A = {'lear': preds['lear'],
           best_linear_run: preds[best_linear_run],
           best_tree_run:   preds[best_tree_run]}
if best_neural_run is not None and best_neural_run in preds.columns:
    preds_A[best_neural_run] = preds[best_neural_run]
preds_A['climatology_how'] = preds['climatology_how']

PA_abs, NA_abs = dm_pairwise_pvalue_matrix(preds_A, preds['actual'], loss='abs')
PA_sq,  NA_sq  = dm_pairwise_pvalue_matrix(preds_A, preds['actual'], loss='sq')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_dm_heatmap(PA_abs, axes[0], 'Matrix A — absolute loss')
plot_dm_heatmap(PA_sq,  axes[1], 'Matrix A — squared loss')
fig.suptitle('Diebold-Mariano (best of each family + LEAR + climatology)', y=1.02, fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_dm_matrix_A.png', bbox_inches='tight')
plt.show()

PA_abs.to_csv(TBL_DIR / 'tbl_dm_matrix_A_abs.csv')
PA_sq.to_csv(TBL_DIR / 'tbl_dm_matrix_A_sq.csv')
NA_abs.to_csv(TBL_DIR / 'tbl_dm_matrix_A_n.csv')
print('Saved Matrix A: tbl_dm_matrix_A_abs.csv, tbl_dm_matrix_A_sq.csv, tbl_dm_matrix_A_n.csv')


In [ ]:
# Matrix B — overall-winning spec across FS variants (mode held at winner's mode)
spec_B = overall_run.split('_')[0]
mode_B = tuned[tuned['run_name'] == overall_run]['ablation_mode'].iloc[0]
ms_B   = MODE_SHORT[mode_B]
preds_B = {f'{spec_B}_{fs}_{ms_B}': preds[f'{spec_B}_{fs}_{ms_B}']
           for fs in FS_ORDER if f'{spec_B}_{fs}_{ms_B}' in preds.columns}

PB_abs, _ = dm_pairwise_pvalue_matrix(preds_B, preds['actual'], loss='abs')
PB_sq,  _ = dm_pairwise_pvalue_matrix(preds_B, preds['actual'], loss='sq')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
plot_dm_heatmap(PB_abs, axes[0], 'Matrix B — absolute loss')
plot_dm_heatmap(PB_sq,  axes[1], 'Matrix B — squared loss')
fig.suptitle(f'Diebold-Mariano — {spec_B} across FS variants  (mode = {mode_B})', y=1.02, fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_dm_matrix_B.png', bbox_inches='tight')
plt.show()

PB_abs.to_csv(TBL_DIR / 'tbl_dm_matrix_B_abs.csv')
PB_sq.to_csv(TBL_DIR / 'tbl_dm_matrix_B_sq.csv')
print('Saved Matrix B: tbl_dm_matrix_B_abs.csv, tbl_dm_matrix_B_sq.csv')


In [ ]:
# Matrix C — overall-winning spec across modes (FS held at winner's FS)
spec_C = overall_run.split('_')[0]
fs_C   = tuned[tuned['run_name'] == overall_run]['feature_variant'].iloc[0]
preds_C = {f'{spec_C}_{fs_C}_{MODE_SHORT[m]}': preds[f'{spec_C}_{fs_C}_{MODE_SHORT[m]}']
           for m in MODE_ORDER if f'{spec_C}_{fs_C}_{MODE_SHORT[m]}' in preds.columns}

PC_abs, _ = dm_pairwise_pvalue_matrix(preds_C, preds['actual'], loss='abs')
PC_sq,  _ = dm_pairwise_pvalue_matrix(preds_C, preds['actual'], loss='sq')

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
plot_dm_heatmap(PC_abs, axes[0], 'Matrix C — absolute loss')
plot_dm_heatmap(PC_sq,  axes[1], 'Matrix C — squared loss')
fig.suptitle(f'Diebold-Mariano — {spec_C} across modes  (FS = {fs_C})', y=1.02, fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_dm_matrix_C.png', bbox_inches='tight')
plt.show()

PC_abs.to_csv(TBL_DIR / 'tbl_dm_matrix_C_abs.csv')
PC_sq.to_csv(TBL_DIR / 'tbl_dm_matrix_C_sq.csv')
print('Saved Matrix C: tbl_dm_matrix_C_abs.csv, tbl_dm_matrix_C_sq.csv')


## 19 — SHAP feature attribution

Cross-model feature importance for the winners of the tree and linear families.
**TreeSHAP** (Lundberg et al. 2020) computes exact Shapley values for tree
ensembles in polynomial time; **LinearSHAP** computes them in closed form for
regularised linear models. SHAP values are in the units of the model output
(EUR/MWh), so they are directly comparable across the two families.

LEAR's per-period Lasso models are not persisted as a single fitted artifact
in this campaign and are excluded. TabNet's KernelSHAP would require a
$\sim$30-minute Monte-Carlo estimation that the user opted to skip; its
attention-mask feature importance is reported separately if needed.


In [ ]:
import pickle
import shap

print('shap', shap.__version__)


def _load_pickle(run_name):
    with open(CACHE / 'models' / f'{run_name}.pkl', 'rb') as f:
        return pickle.load(f)


def _features_of(model_obj):
    if hasattr(model_obj, 'feature_names_in_'):
        return list(model_obj.feature_names_in_)
    if hasattr(model_obj, 'get_booster'):
        return model_obj.get_booster().feature_names
    raise RuntimeError('cannot infer feature names from model')


df_final = pd.read_parquet(CACHE / 'df_final.parquet')
TARGET   = 'entso_e.dk1.price_da_pt15m'
test_idx = preds.index

print(f'Tree winner    : {best_tree_run}')
print(f'Linear winner  : {best_linear_run}')
print(f'X test window  : {len(test_idx)} rows')


In [ ]:
# ----- TreeSHAP for the tree-family winner
tree_model = _load_pickle(best_tree_run)
tree_feats = _features_of(tree_model)
X_tree = df_final.loc[test_idx, tree_feats]

explainer_tree = shap.TreeExplainer(tree_model)
shap_tree = explainer_tree(X_tree)
mean_abs_tree = (pd.Series(np.abs(shap_tree.values).mean(axis=0), index=tree_feats)
                   .sort_values(ascending=False))

# Beeswarm (max 20 features)
plt.figure(figsize=(7.8, 6))
shap.plots.beeswarm(shap_tree, max_display=20, show=False)
plt.title(f'SHAP beeswarm — {best_tree_run}  (n={len(X_tree)})', fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_beeswarm_{best_tree_run}.png', bbox_inches='tight'); plt.show()

# mean(|SHAP|) bar (top 20)
plt.figure(figsize=(7.8, 6))
shap.plots.bar(shap_tree, max_display=20, show=False)
plt.title(f'mean(|SHAP|) — {best_tree_run}', fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_bar_{best_tree_run}.png', bbox_inches='tight'); plt.show()

# Top-3 dependence plots
top3_tree = mean_abs_tree.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for k, feat in enumerate(top3_tree):
    j = tree_feats.index(feat)
    ax = axes[k]
    ax.scatter(X_tree[feat].values, shap_tree.values[:, j], s=4, alpha=0.4, color='#1f77b4')
    ax.axhline(0, color='black', linewidth=0.6, alpha=0.5)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('SHAP value (EUR/MWh)', fontsize=8)
    ax.set_title(f'top-{k+1}', fontsize=10)
fig.suptitle(f'SHAP dependence — {best_tree_run}', y=1.04, fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_dep_{best_tree_run}.png', bbox_inches='tight'); plt.show()

mean_abs_tree.head(20).rename('mean_abs_shap').to_csv(TBL_DIR / f'tbl_shap_top_{best_tree_run}.csv')


In [ ]:
# ----- LinearSHAP for the linear-family winner (sklearn Pipeline: scaler -> model)
linear_pipe   = _load_pickle(best_linear_run)
linear_feats  = _features_of(linear_pipe)
X_linear_raw  = df_final.loc[test_idx, linear_feats]

# Walk the pipeline so the final estimator gets its expected scaled inputs
X_pre = X_linear_raw.copy()
for _, step in linear_pipe.steps[:-1]:
    X_pre = pd.DataFrame(step.transform(X_pre), columns=linear_feats, index=X_pre.index)
final_estimator = linear_pipe.steps[-1][1]

masker = shap.maskers.Independent(X_pre.sample(min(200, len(X_pre)), random_state=0), max_samples=200)
explainer_lin = shap.LinearExplainer(final_estimator, masker)
shap_lin = explainer_lin(X_pre)
mean_abs_lin = (pd.Series(np.abs(shap_lin.values).mean(axis=0), index=linear_feats)
                  .sort_values(ascending=False))

plt.figure(figsize=(7.8, 6))
shap.plots.beeswarm(shap_lin, max_display=20, show=False)
plt.title(f'SHAP beeswarm — {best_linear_run}  (n={len(X_pre)})', fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_beeswarm_{best_linear_run}.png', bbox_inches='tight'); plt.show()

plt.figure(figsize=(7.8, 6))
shap.plots.bar(shap_lin, max_display=20, show=False)
plt.title(f'mean(|SHAP|) — {best_linear_run}', fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_bar_{best_linear_run}.png', bbox_inches='tight'); plt.show()

# For the linear model, plot SHAP vs the *original* (unscaled) feature value —
# more interpretable than scaled units. The relationship is linear by construction.
top3_lin = mean_abs_lin.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for k, feat in enumerate(top3_lin):
    j = linear_feats.index(feat)
    ax = axes[k]
    ax.scatter(X_linear_raw[feat].values, shap_lin.values[:, j], s=4, alpha=0.4, color='#d62728')
    ax.axhline(0, color='black', linewidth=0.6, alpha=0.5)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('SHAP value (EUR/MWh)', fontsize=8)
    ax.set_title(f'top-{k+1}', fontsize=10)
fig.suptitle(f'SHAP dependence — {best_linear_run}', y=1.04, fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR / f'fig_shap_dep_{best_linear_run}.png', bbox_inches='tight'); plt.show()

mean_abs_lin.head(20).rename('mean_abs_shap').to_csv(TBL_DIR / f'tbl_shap_top_{best_linear_run}.csv')


In [ ]:
# ----- Cross-model comparison: union of top-10 features per model
top10_tree = mean_abs_tree.head(10).index.tolist()
top10_lin  = mean_abs_lin.head(10).index.tolist()
union_feats = list(dict.fromkeys(top10_tree + top10_lin))

cross = pd.DataFrame({
    best_tree_run:   mean_abs_tree.reindex(union_feats).values,
    best_linear_run: mean_abs_lin.reindex(union_feats).values,
}, index=union_feats)

fig, ax = plt.subplots(figsize=(10, max(4.5, 0.36 * len(union_feats))))
y = np.arange(len(union_feats))
ax.barh(y - 0.18, cross[best_tree_run].fillna(0),   height=0.36, label=best_tree_run,   color='#1f77b4')
ax.barh(y + 0.18, cross[best_linear_run].fillna(0), height=0.36, label=best_linear_run, color='#d62728')
ax.set_yticks(y); ax.set_yticklabels(union_feats, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('mean |SHAP|  (EUR/MWh)')
ax.set_title('Cross-model feature importance — union of top-10 per model')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_shap_cross_model.png', bbox_inches='tight')
plt.show()

cross.to_csv(TBL_DIR / 'tbl_shap_cross_model.csv')
print('Saved cross-model SHAP table to', TBL_DIR / 'tbl_shap_cross_model.csv')
print()
print('SHAP coverage:')
print(f'  ✓ tree winner   {best_tree_run}    — TreeSHAP (exact)')
print(f'  ✓ linear winner {best_linear_run}  — LinearSHAP (exact)')
print(f'  – LEAR              — fitted per-period Lassos not persisted; excluded')
print(f'  – TabNet            — KernelSHAP skipped (compute cost); excluded')


## 20 — Render LaTeX table fragments (chapter + appendix)

The cell below converts the result CSVs in `_build/figures/tables/` into
booktabs-formatted `.tex` fragments suitable for `\input{}` in the thesis
source. Two output flavours are produced for every table that benefits from
the split:

* **chapter** versions are concise (top-15 rows, or filtered to the winning
  spec) — meant for the body of the Results chapter.
* **appendix** versions are exhaustive (all rows) — meant for the Results
  appendix where the full experimental matrix is reported.

Tables that are already short (best-of-family, FS deltas, DM matrices,
cross-model SHAP) ship a single version that is referenced from both the
chapter and the appendix.


In [ ]:
"""Render LaTeX table fragments for the thesis Results chapter and appendix.

Reads CSVs from `_build/figures/tables/` and writes booktabs `.tex`
fragments next to them. Each fragment is the bare `tabular` environment —
the thesis source supplies `\begin{table}`, caption, and label around the
`\input{...}`.
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Iterable, Optional

import pandas as pd

# Path resolution: this notebook lives at notebooks/thesis/, the build
# directory at notebooks/thesis/_build/figures/tables/. Use cwd + relative
# so the cell works whether or not the kernel was launched from the
# notebook directory.
def _locate_tables_dir():
    rel = Path("_build/figures/tables")
    here = Path.cwd().resolve()
    candidates = [here / rel]
    for parent in here.parents:
        candidates.append(parent / rel)
        candidates.append(parent / "notebooks/thesis" / rel)
    candidates.append(PACKAGE_ROOT / "figures/tables")
    for c in candidates:
        if c.exists() and (c / "tbl_headline.csv").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not locate _build/figures/tables. Run the rest of the "
        "notebook first so the result CSVs are populated."
    )


TABLES_DIR = _locate_tables_dir()

# ----------------------------------------------------------------------
# LaTeX escape + cell formatters
# ----------------------------------------------------------------------

_LATEX_ESC = {
    "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
    "_": r"\_", "{": r"\{", "}": r"\}",
    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    "\\": r"\textbackslash{}",
}


def latex_escape(s):
    if s is None:
        return "---"
    if isinstance(s, float) and pd.isna(s):
        return "---"
    return "".join(_LATEX_ESC.get(c, c) for c in str(s))


def _is_missing(v):
    if v is None:
        return True
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):
        return False


def fmt_float(decimals=2, dash="---"):
    def f(v):
        if _is_missing(v):
            return dash
        return f"{float(v):.{decimals}f}"
    return f


def fmt_int(v, dash="---"):
    if _is_missing(v):
        return dash
    return f"{int(v)}"


def fmt_pvalue(v, dash="---"):
    """Bold p<0.01, underline 0.01<=p<0.05, plain otherwise. Cap below 1e-3."""
    if _is_missing(v):
        return dash
    x = float(v)
    if x < 1e-3:
        return r"$<\!10^{-3}$"
    if x < 0.01:
        return r"\textbf{" + f"{x:.3f}" + "}"
    if x < 0.05:
        return r"\underline{" + f"{x:.3f}" + "}"
    return f"{x:.3f}"


def fmt_run(v):
    if _is_missing(v):
        return "---"
    return r"\texttt{" + latex_escape(v) + "}"


def fmt_str(v):
    return latex_escape(v)


def fmt_small_run(v):
    if _is_missing(v):
        return "---"
    return r"\texttt{\small " + latex_escape(v) + "}"


# ----------------------------------------------------------------------
# Core renderers
# ----------------------------------------------------------------------

def render_tabular(df, *, formatters=None, headers=None, column_format=None,
                    bold_rows=None, midrule_after=None):
    work = df.reset_index(drop=True).copy()
    formatters = formatters or {}

    out_cols = {}
    for col in work.columns:
        if col in formatters:
            out_cols[col] = work[col].apply(formatters[col])
        else:
            out_cols[col] = work[col].apply(latex_escape)
    work = pd.DataFrame(out_cols)

    if headers is None:
        headers = [latex_escape(c) for c in work.columns]
    if column_format is None:
        column_format = " ".join(["l"] * len(headers))

    bold_rows_set = set(bold_rows or [])
    midrule_set = set(midrule_after or [])

    n_rows = len(work)
    lines = []
    lines.append(r"\begin{tabular}{" + column_format + "}")
    lines.append(r"\toprule")
    lines.append(" & ".join(headers) + r" \\")
    lines.append(r"\midrule")
    for i in range(n_rows):
        cells = [str(work.iloc[i, j]) for j in range(work.shape[1])]
        if i in bold_rows_set:
            cells = [r"\textbf{" + c + "}" for c in cells]
        lines.append(" & ".join(cells) + r" \\")
        if i in midrule_set and i < n_rows - 1:
            lines.append(r"\midrule")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)


def render_square(df, *, cell_formatter, diag_text="---",
                   row_label_formatter=fmt_run, col_label_formatter=fmt_run,
                   column_align="c"):
    work = df.copy()
    label_col = work.columns[0]
    work = work.set_index(label_col)

    cols = list(work.columns)
    headers = [""] + [col_label_formatter(c) for c in cols]
    column_format = "l" + (" " + column_align) * len(cols)

    lines = []
    lines.append(r"\begin{tabular}{" + column_format + "}")
    lines.append(r"\toprule")
    lines.append(" & ".join(headers) + r" \\")
    lines.append(r"\midrule")
    for row_label in work.index:
        cells = [row_label_formatter(row_label)]
        for col in cols:
            v = work.at[row_label, col]
            if str(row_label) == str(col):
                cells.append(diag_text)
            else:
                cells.append(cell_formatter(v))
        lines.append(" & ".join(cells) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)


def write_tex(name, fragment):
    out = TABLES_DIR / name
    out.write_text(fragment)
    return out


# ----------------------------------------------------------------------
# Per-table renderers
# ----------------------------------------------------------------------

def _headline_columns():
    return [
        "rank", "run_name", "family", "feature_variant", "ablation_mode",
        "test_mae", "test_rmse", "test_r2",
        "n_selected_groups", "n_selected_features", "study_wall_seconds",
    ]


def _headline_formatters():
    return {
        "rank": fmt_int,
        "run_name": fmt_run,
        "family": fmt_str,
        "feature_variant": lambda v: latex_escape(str(v).upper()),
        "ablation_mode": lambda v: latex_escape(str(v).replace("_", " ")),
        "test_mae": fmt_float(2),
        "test_rmse": fmt_float(2),
        "test_r2": fmt_float(3),
        "n_selected_groups": fmt_int,
        "n_selected_features": fmt_int,
        "study_wall_seconds": fmt_int,
    }


_HEADLINE_HEADERS = [
    "Rank", "Run", "Family", "FS", "Mode",
    "MAE", "RMSE", "$R^2$",
    r"\#\,Groups", r"\#\,Features", "Wall (s)",
]
_HEADLINE_COLFMT = "r l l l l r r r r r r"


def render_headline(out_name, n):
    df = pd.read_csv(TABLES_DIR / "tbl_headline.csv")
    if n is not None:
        df = df.head(n)
    fragment = render_tabular(
        df[_headline_columns()],
        formatters=_headline_formatters(),
        headers=_HEADLINE_HEADERS,
        column_format=_HEADLINE_COLFMT,
        bold_rows=[0],
    )
    return write_tex(out_name, fragment)


def render_best_vs_baselines():
    family = pd.read_csv(TABLES_DIR / "tbl_best_per_family.csv")
    baselines = pd.read_csv(TABLES_DIR / "tbl_baselines_vs_best.csv")
    baselines = baselines[baselines["kind"] == "baseline"].copy()

    rows = []
    for _, r in family.iterrows():
        rows.append({
            "kind": str(r["family"]).capitalize() + " (best)",
            "name": r["run_name"],
            "fs": str(r["feature_variant"]).upper(),
            "mode": str(r["ablation_mode"]).replace("_", " "),
            "test_mae": r["test_mae"],
            "test_rmse": r["test_rmse"],
        })
    for _, r in baselines.iterrows():
        rows.append({
            "kind": "Baseline",
            "name": r["name"],
            "fs": None,
            "mode": None,
            "test_mae": r["test_mae"],
            "test_rmse": r["test_rmse"],
        })

    df = pd.DataFrame(rows)
    lear_mae = float(df.loc[df["name"] == "lear", "test_mae"].iloc[0])
    df["rmae_lear"] = df["test_mae"] / lear_mae

    fam_part = df[df["kind"].str.endswith("(best)")].sort_values("test_mae").reset_index(drop=True)
    base_part = df[df["kind"] == "Baseline"].sort_values("test_mae").reset_index(drop=True)
    df = pd.concat([fam_part, base_part], ignore_index=True)
    n_family = len(fam_part)

    fragment = render_tabular(
        df,
        formatters={
            "kind": fmt_str, "name": fmt_run, "fs": fmt_str, "mode": fmt_str,
            "test_mae": fmt_float(2), "test_rmse": fmt_float(2),
            "rmae_lear": fmt_float(2),
        },
        headers=["Kind", "Name", "FS", "Mode", "MAE", "RMSE", "rMAE"],
        column_format="l l l l r r r",
        bold_rows=[0],
        midrule_after=[n_family - 1],
    )
    return write_tex("tbl_best_vs_baselines.tex", fragment)


def render_dm_matrix(matrix_letter, loss):
    df = pd.read_csv(TABLES_DIR / f"tbl_dm_matrix_{matrix_letter}_{loss}.csv")
    fragment = render_square(df, cell_formatter=fmt_pvalue)
    return write_tex(f"tbl_dm_matrix_{matrix_letter}_{loss}.tex", fragment)


def render_dm_n_matrix(matrix_letter="A"):
    df = pd.read_csv(TABLES_DIR / f"tbl_dm_matrix_{matrix_letter}_n.csv")
    fragment = render_square(df, cell_formatter=fmt_int, diag_text="---")
    return write_tex(f"tbl_dm_matrix_{matrix_letter}_n.tex", fragment)


_THREEMODE_FORMATTERS = {
    "model": fmt_str,
    "feature_variant": lambda v: latex_escape(str(v).upper()),
    "fixed": fmt_float(2),
    "no_penalty": fmt_float(2),
    "with_penalty": fmt_float(2),
    "Δ(np−fixed)": fmt_float(2),
    "Δ(wp−fixed)": fmt_float(2),
    "Δ(wp−np)": fmt_float(2),
    "best_mode": lambda v: latex_escape(str(v).replace("_", " ")),
}
_THREEMODE_HEADERS = [
    "Model", "FS", "Fixed", "No-pen.", "With-pen.",
    r"$\Delta_\text{np}$", r"$\Delta_\text{wp}$",
    r"$\Delta_{wp-np}$", "Best",
]
_THREEMODE_COLFMT = "l l r r r r r r l"


def render_three_mode_full():
    df = pd.read_csv(TABLES_DIR / "tbl_three_mode.csv")
    fragment = render_tabular(
        df, formatters=_THREEMODE_FORMATTERS,
        headers=_THREEMODE_HEADERS, column_format=_THREEMODE_COLFMT,
    )
    return write_tex("tbl_three_mode_full.tex", fragment)


def render_three_mode_filter(model="lgbm"):
    df = pd.read_csv(TABLES_DIR / "tbl_three_mode.csv")
    df = df[df["model"] == model].reset_index(drop=True)
    fragment = render_tabular(
        df, formatters=_THREEMODE_FORMATTERS,
        headers=_THREEMODE_HEADERS, column_format=_THREEMODE_COLFMT,
    )
    return write_tex(f"tbl_three_mode_{model}.tex", fragment)


def render_fs_deltas():
    df = pd.read_csv(TABLES_DIR / "tbl_fs_deltas.csv")
    fragment = render_tabular(
        df,
        formatters={
            "model": fmt_str,
            "Δ(fs2−fs1)": fmt_float(2),
            "Δ(fs3−fs2)": fmt_float(2),
            "Δ(fs4−fs3)": fmt_float(2),
            "Δ(fs4−fs1)": fmt_float(2),
        },
        headers=["Model",
                 r"$\Delta_{2-1}$", r"$\Delta_{3-2}$",
                 r"$\Delta_{4-3}$", r"$\Delta_{4-1}$"],
        column_format="l r r r r",
    )
    return write_tex("tbl_fs_deltas.tex", fragment)


_PARSIMONY_FORMATTERS = {
    "model": fmt_str,
    "feature_variant": lambda v: latex_escape(str(v).upper()),
    "n_selected_groups__no_penalty": fmt_int,
    "n_selected_groups__with_penalty": fmt_int,
    "n_selected_features__no_penalty": fmt_int,
    "n_selected_features__with_penalty": fmt_int,
    "Δ_groups (wp−np)": fmt_float(0),
    "Δ_features (wp−np)": fmt_float(0),
}
_PARSIMONY_HEADERS = [
    "Model", "FS",
    r"\#G\,(np)", r"\#G\,(wp)",
    r"\#F\,(np)", r"\#F\,(wp)",
    r"$\Delta$\,G", r"$\Delta$\,F",
]
_PARSIMONY_COLFMT = "l l r r r r r r"


def render_parsimony_full():
    df = pd.read_csv(TABLES_DIR / "tbl_parsimony.csv")
    fragment = render_tabular(
        df, formatters=_PARSIMONY_FORMATTERS,
        headers=_PARSIMONY_HEADERS, column_format=_PARSIMONY_COLFMT,
    )
    return write_tex("tbl_parsimony_full.tex", fragment)


def render_parsimony_filter(model="lgbm"):
    df = pd.read_csv(TABLES_DIR / "tbl_parsimony.csv")
    df = df[df["model"] == model].reset_index(drop=True)
    fragment = render_tabular(
        df, formatters=_PARSIMONY_FORMATTERS,
        headers=_PARSIMONY_HEADERS, column_format=_PARSIMONY_COLFMT,
    )
    return write_tex(f"tbl_parsimony_{model}.tex", fragment)


def render_group_frequency(out_name, n):
    df = pd.read_csv(TABLES_DIR / "tbl_group_frequency_corrected.csv")
    df = df.rename(columns={"Unnamed: 0": "alg_group"})
    df = df.sort_values("relative_freq", ascending=False)
    if n is not None:
        df = df.head(n)
    df = df.reset_index(drop=True)
    fragment = render_tabular(
        df[["alg_group", "family", "selected", "candidate", "relative_freq", "raw_pct_of_alg"]],
        formatters={
            "alg_group": fmt_small_run, "family": fmt_str,
            "selected": fmt_int, "candidate": fmt_int,
            "relative_freq": fmt_float(3), "raw_pct_of_alg": fmt_float(1),
        },
        headers=["Group", "Family", "Sel.", "Cand.",
                 "Rel.\\ freq.", r"\%\,of cells"],
        column_format="l l r r r r",
    )
    return write_tex(out_name, fragment)


def render_shap_top(model_run, out_name, n):
    df = pd.read_csv(TABLES_DIR / f"tbl_shap_top_{model_run}.csv")
    df = df.rename(columns={"Unnamed: 0": "feature"})
    if n is not None:
        df = df.head(n)
    df = df.reset_index(drop=True)
    df.insert(0, "rank", range(1, len(df) + 1))
    fragment = render_tabular(
        df,
        formatters={
            "rank": fmt_int, "feature": fmt_small_run,
            "mean_abs_shap": fmt_float(3),
        },
        headers=["Rank", "Feature", r"$\overline{|\phi|}$"],
        column_format="r l r",
    )
    return write_tex(out_name, fragment)


def render_shap_cross_model():
    df = pd.read_csv(TABLES_DIR / "tbl_shap_cross_model.csv")
    df = df.rename(columns={"Unnamed: 0": "feature"})
    fragment = render_tabular(
        df,
        formatters={
            "feature": fmt_small_run,
            "lgbm_fs4_wp": fmt_float(3),
            "lasso_fs3_wp": fmt_float(3),
        },
        headers=["Feature",
                 r"\texttt{lgbm\_fs4\_wp}", r"\texttt{lasso\_fs3\_wp}"],
        column_format="l r r",
    )
    return write_tex("tbl_shap_cross_model.tex", fragment)


def render_rmae(out_name, n):
    df = pd.read_csv(TABLES_DIR / "tbl_rmae.csv")
    if n is not None:
        df = df.head(n)
    fragment = render_tabular(
        df[["rank", "run_name", "family", "feature_variant", "ablation_mode",
            "test_mae", "rMAE_lear", "rMAE_climatology",
            "rMAE_naive_d1", "rMAE_naive_w1", "rMAE_epf_hybrid"]],
        formatters={
            "rank": fmt_int, "run_name": fmt_run, "family": fmt_str,
            "feature_variant": lambda v: latex_escape(str(v).upper()),
            "ablation_mode": lambda v: latex_escape(str(v).replace("_", " ")),
            "test_mae": fmt_float(2),
            "rMAE_lear": fmt_float(2), "rMAE_climatology": fmt_float(2),
            "rMAE_naive_d1": fmt_float(2), "rMAE_naive_w1": fmt_float(2),
            "rMAE_epf_hybrid": fmt_float(2),
        },
        headers=["Rank", "Run", "Family", "FS", "Mode", "MAE",
                 "rMAE LEAR", "rMAE Clim.",
                 r'rMAE Na\"ive\textsubscript{d}',
                 r'rMAE Na\"ive\textsubscript{w}',
                 "rMAE EPF"],
        column_format="r l l l l r r r r r r",
        bold_rows=[0],
    )
    return write_tex(out_name, fragment)


# ----------------------------------------------------------------------
# Driver
# ----------------------------------------------------------------------

def render_all(scope="both"):
    """scope: 'chapter' | 'appendix' | 'both' — controls which subsets render."""
    if scope not in ("chapter", "appendix", "both"):
        raise ValueError(scope)
    out = {}

    out["best_vs_baselines"] = render_best_vs_baselines()
    out["dm_A_abs"] = render_dm_matrix("A", "abs")
    out["dm_A_sq"] = render_dm_matrix("A", "sq")
    out["dm_A_n"] = render_dm_n_matrix("A")
    out["dm_B_abs"] = render_dm_matrix("B", "abs")
    out["dm_B_sq"] = render_dm_matrix("B", "sq")
    out["dm_C_abs"] = render_dm_matrix("C", "abs")
    out["dm_C_sq"] = render_dm_matrix("C", "sq")
    out["fs_deltas"] = render_fs_deltas()
    out["shap_cross"] = render_shap_cross_model()

    if scope in ("chapter", "both"):
        out["headline_top15"]  = render_headline("tbl_headline_top15.tex", 15)
        out["three_mode_lgbm"] = render_three_mode_filter("lgbm")
        out["parsimony_lgbm"]  = render_parsimony_filter("lgbm")
        out["group_freq_top20"] = render_group_frequency("tbl_group_frequency_top20.tex", 20)
        out["shap_top_lgbm"]   = render_shap_top("lgbm_fs4_wp", "tbl_shap_top_lgbm_fs4_wp.tex", 15)
        out["shap_top_lasso"]  = render_shap_top("lasso_fs3_wp", "tbl_shap_top_lasso_fs3_wp.tex", 15)
        out["rmae_top15"]      = render_rmae("tbl_rmae_top15.tex", 15)

    if scope in ("appendix", "both"):
        out["headline_full"]   = render_headline("tbl_headline_full.tex", None)
        out["three_mode_full"] = render_three_mode_full()
        out["parsimony_full"]  = render_parsimony_full()
        out["group_freq_full"] = render_group_frequency("tbl_group_frequency_full.tex", None)
        out["shap_full_lgbm"]  = render_shap_top("lgbm_fs4_wp", "tbl_shap_full_lgbm_fs4_wp.tex", None)
        out["shap_full_lasso"] = render_shap_top("lasso_fs3_wp", "tbl_shap_full_lasso_fs3_wp.tex", None)
        out["rmae_full"]       = render_rmae("tbl_rmae_full.tex", None)

    return out


# Render both scopes by default. Edit scope= to "chapter" or "appendix" if
# you want only one set.
_paths = render_all(scope="both")
print(f"Rendered {len(_paths)} LaTeX fragments to {TABLES_DIR}")
for label, p in sorted(_paths.items()):
    rows = p.read_text().count(" \\\\")
    print(f"  {label:24s} -> {p.name:40s} ({rows-1:3d} rows)")
